# 2. Diseño experimental

*Autores: María Carolina Cantillo Orozco (200179105) y Juan Camilo Oñoro Araujo (200177329)*

```{admonition} Qué responde este capítulo
:class: tip
**Pregunta.** ¿Cómo se entrenan y evalúan 156 combinaciones de forma justa, sin fuga de información y en un
portátil?

**Respuesta corta.** Todas pasan por la misma validación anidada (5 pliegues externos × 3 internos, armados con
kNNDM para imitar zonas nuevas), con el preprocesamiento y el balanceo dentro de cada pliegue, 12 evaluaciones
de búsqueda y una submuestra estratificada de 10 001 viviendas. Cada una de las 640 tareas queda registrada en
una tabla maestra.
```

## 2.1 Un pipeline combinatorio

La guía pide entrenar el **producto** de alternativas, no su suma: cada modelo con cada técnica de balanceo y
cada optimizador. A los 7 modelos de clasificación de la guía se agrega el **XGBoost geográfico**, el modelo
nuevo del proyecto, para medirlo con exactamente el mismo protocolo.

```{note}
**Propuesta y referencias.** Los 7 modelos de la guía no son la propuesta del proyecto: son los modelos de referencia
(*benchmarks*) contra los que se pone a prueba el XGBoost geográfico, el único que no se había usado antes para
clasificar el estrato (revisión bibliográfica del Entregable 2). La introducción explica el papel de cada uno.
```

| Tarea | Modelos | Balanceo | Optimización | Total |
|---|---|---|---|---|
| Clasificación | 7 + 1 (XGBoost geográfico) | 4 | 4 | 7 × 4 × 4 = **112** (+ 16) |
| Regresión | 7 | no aplica | 4 | 7 × 4 = **28** |
| **Total** | | | | **140** (+ 16) |

El pipeline está programado con funciones parametrizadas (modelo → balanceo → optimizador) y cada corrida se
registra en una tabla maestra (sección 2.5).

**Equivalentes de `class_weight='balanced'`.** Naive Bayes no tiene pesos de clase: se usan priors uniformes,
que es el mismo efecto. KNN tampoco: se divide cada probabilidad por la frecuencia de la clase en el
entrenamiento y se renormaliza ($\tilde p_k \propto p_k/\pi_k$), la corrección bayesiana que convierte priors
empíricos en uniformes. XGBoost y el geográfico usan pesos de muestra $1/\text{frecuencia}$.

**SMOTE y ADASYN** van dentro de `imblearn.pipeline.Pipeline`, después del preprocesamiento: se aplican solo al
entrenamiento de cada pliegue y nunca al predecir. ADASYN reparte las muestras sintéticas según cuántos vecinos
de otra clase tiene cada ejemplo; cuando un estrato no tiene ninguno (algo frecuente con un estrato tan
concentrado), su densidad es cero y la librería falla. En ese caso se aplica estrato por estrato y el que falla
se deja sin muestras sintéticas.

## 2.2 Validación cruzada anidada con kNNDM

Elegir hiperparámetros mirando el mismo esquema con que se reporta el desempeño da una cifra optimista. Por
eso:

- **Bucle externo (5 pliegues):** estima el desempeño de generalización. Sus pliegues se arman con **kNNDM**
  (*k-fold nearest neighbour distance matching*; Linnenbrink et al., 2024): grupos compactos de K-Means
  repartidos en pliegues de modo que la distancia de cada vivienda de validación a la vivienda de entrenamiento
  más cercana se parezca a la del escenario real (la geometría del test del Entregable 1). Las unidades de un
  mismo edificio nunca se separan (ICC 0.99).
- **Bucle interno (3 pliegues):** dentro del entrenamiento de cada pliegue externo, otro kNNDM cuyo escenario
  es el propio pliegue externo: la búsqueda de hiperparámetros se valida a la misma distancia a la que luego se
  evalúa.

**Por qué 5 × 3.** Con 5 pliegues externos hay una media y una desviación entre pliegues (sección 5.4) y bloques
para la prueba de Friedman. Con 3 internos cada evaluación cuesta 3 ajustes; 5 internos habrían multiplicado el
costo por 5/3 sin cambiar la elección del óptimo en el piloto.

## 2.3 Una submuestra para la búsqueda

Con las 257 273 viviendas de entrenamiento, las 156 combinaciones con validación anidada costaban unas 630
horas-CPU en el portátil del proyecto (Intel Core i5-8265U, 4 núcleos; piloto del 18-oct). Se usa una
**submuestra estratificada de 10 000 viviendas** (mismas proporciones de estrato), con 12 evaluaciones por
búsqueda: 38 horas de cómputo en tareas, que con 8 procesos cupieron en 4.8 horas. Los finalistas se reentrenan
y se evalúan en el test del Entregable 1, y además se reentrenan con las 257 273 viviendas para medir cuánto
cuesta la submuestra (sección 9.4).

In [1]:
import pickle

import numpy as np
import pandas as pd

from _comun import RES, leer_csv

with open(RES / "datos_clasificacion.pkl", "rb") as fh:
    paq = pickle.load(fh)
sub = paq["sub"]
print(f"Submuestra: {len(sub):,} viviendas | kNNDM externo: q = {paq['knndm']['q']} grupos, "
      f"Wasserstein {paq['knndm']['W_km']:.3f} km (validación aleatoria: {paq['knndm']['W_aleatoria_km']:.3f} km)")
pd.DataFrame({"pliegue externo": range(5),
              "entrenamiento": [len(a) for a, _ in paq["externos"]],
              "validación": [len(b) for _, b in paq["externos"]],
              "estratos en validación": [len(np.unique(sub.estrato_num.to_numpy()[b])) for _, b in paq["externos"]],
              "pliegues internos (tamaños de validación)": [", ".join(str(len(v)) for _, v in i) for i in paq["internos"]]})

Submuestra: 10,001 viviendas | kNNDM externo: q = 50 grupos, Wasserstein 0.124 km (validación aleatoria: 0.431 km)


,pliegue externo,entrenamiento,validación,estratos en validación,pliegues internos (tamaños de validación)
0,0,7943,2058,6,"2639, 2664, 2640"
1,1,8128,1873,6,"2729, 2683, 2716"
2,2,7990,2011,6,"2684, 2688, 2618"
3,3,7965,2036,6,"2602, 2569, 2794"
4,4,7978,2023,6,"2701, 2643, 2634"


In [2]:
(sub.estrato_num.value_counts(normalize=True).sort_index().rename("proporción en la submuestra").to_frame()
 .style.format("{:.1%}"))

,proporción en la submuestra
estrato_num,
1,30.0%
2,22.7%
3,22.9%
4,14.4%
5,5.2%
6,4.8%


## 2.4 Semilla única

Una sola semilla global (`SEED = 42`, en `estrato3/configuracion.py`) se propaga a numpy, scikit-learn,
imbalanced-learn (SMOTE/ADASYN), XGBoost, Optuna (TPE) y DEAP. La sensibilidad a otras semillas se mide en el
capítulo 5.

## 2.5 Tabla maestra de experimentos

Cada tarea (combinación × pliegue externo) guarda su semilla, sus hiperparámetros finales, las métricas del
bucle externo, el tiempo de búsqueda, de ajuste y de inferencia, el número de evaluaciones y el historial
completo de la búsqueda. La tabla maestra las reúne:

In [3]:
tm = leer_csv("tabla_maestra_clasificacion.csv")
if tm is not None:
    print(f"{len(tm)} tareas registradas ({tm.groupby(['modelo', 'balanceo', 'metodo']).ngroups} combinaciones)")
    display(tm[["modelo", "balanceo", "metodo", "fold", "f1_macro", "auc_ovr", "n_evaluaciones", "seg_busqueda",
                "params"]].head(12))

640 tareas registradas (128 combinaciones)


,modelo,balanceo,metodo,fold,f1_macro,auc_ovr,n_evaluaciones,seg_busqueda,params
0,SVM,ninguno,Bayesiana,2,0.517046,0.869238,12,189.724244,"{""gamma"": 0.007188274526338213, ""n_components""..."
1,SVM,ninguno,Grid,0,0.605863,0.921779,12,220.615410,"{""gamma"": 0.03, ""n_components"": 300, ""C"": 10.0}"
2,Logística L1/L2,smote,Genética,1,0.740358,0.931903,12,232.473543,"{""penalty"": ""l2"", ""C"": 0.03478056878034407}"
3,XGBoost geográfico,smote,Bayesiana,0,0.642805,0.936005,12,435.170296,"{""n_estimators"": 182, ""learning_rate"": 0.06443..."
4,Naive Bayes,class_weight,Genética,2,0.291275,0.750880,12,8.613736,"{""var_smoothing"": 0.06108254627409418}"
5,XGBoost,class_weight,Genética,3,0.622791,0.927669,12,176.482811,"{""n_estimators"": 152, ""learning_rate"": 0.08241..."
6,XGBoost geográfico,ninguno,Bayesiana,2,0.589674,0.895117,12,222.002559,"{""n_estimators"": 141, ""learning_rate"": 0.12195..."
7,Árbol de decisión,ninguno,Grid,3,0.546517,0.875510,12,13.558258,"{""max_depth"": 10, ""min_samples_leaf"": 50, ""cri..."
8,SVM,smote,Grid,4,0.629443,0.915162,12,451.349229,"{""gamma"": 0.03, ""n_components"": 300, ""C"": 0.1}"
9,Naive Bayes,class_weight,Genética,0,0.373592,0.865896,12,10.814411,"{""var_smoothing"": 0.0999999999999999}"
